# Data Processing

In [33]:
import pandas as pd
import os

In [34]:
analysis_positions = [1, 3, 5, 7, 9]

manual_path = "/home/vilota/566-qa-2/621D/IMG/2号机汇总结果数据检 - mingjie(结果数据).csv"

df = pd.read_csv(manual_path)
df = df.fillna("o")

df = df.rename(columns={"????": "Capture Time", "??": "Grade", "NG??": "NG", "???": "SN", "??.1": "Short SN Pairing"})
column_to_drop = ['Capture Time', 'Grade', 'NG', 'Short SN Pairing', 'Full SN', '???.1', '???.2', '???_Focus', '???_Focus.1', 'AA?_??S', 'AA?_??S2', 'AA?_??T', 'AA?_??T2', 'AA?_??S.1', 'AA?_??T.1', 'AA?_??S.2', 'AA?_??T.2', 'AA?_??S.3', 'AA?_??T.3', 'AA?_??S.4', 'AA?_??T.4', 'AA?_Tilt-X', 'AA?_Tilt-Y', 'AA?_OC-X', 'AA?_OC-Y', 'AA?_???X', 'AA?_???Y', 'AA?_??0.5F-S', 'AA?_??0.5F-T', 'AA?_??0.5F-S.1', 'AA?_??0.5F-T.1', 'AA?_??0.5F-S.2', 'AA?_??0.5F-T.2', 'AA?_??0.5F-S.3', 'AA?_??0.5F-T.3', 'AA?_????', 'AA?_????.1', 'AA?_????.2', 'AA?_????.3', 'AA?_??0.5??', 'AA?_??0.5??.1', 'AA?_??0.5??.2', 'AA?_??0.5??.3']
df = df.drop(columns=column_to_drop, errors='ignore')

pos_columns = ['pos 1', 'pos 3', 'pos 5', 'pos 7', 'pos 9']

collapse_map = {
    'xf': 'f',
    'mf': 'f',
    'sf': 'o',
    'mn': 'n',
    'xn': 'n',
}

allowed_labels = {'f', 'o', 'sn', 'n'}

for col in pos_columns:
    if col in df.columns:
        df[col] = (
            df[col].astype(str).str.split('/').str[0].str.strip().str.lower().str.replace(r'\\.0$', '', regex=True)
        )
        df[col] = df[col].replace(collapse_map)
        df[col] = df[col].where(df[col].isin(allowed_labels))

df = df.dropna(subset=pos_columns, how='all')

df['SN'] = df['SN'].astype(str).str.split('/').str[0].str.strip().str.upper().str.replace(r'\\.0$', '', regex=True)


In [35]:
df.head()

,SN,pos 5,pos 1,pos 3,pos 7,pos 9
0,2710,o,sn,o,n,sn
1,2712,o,o,o,o,o
2,2713,o,o,o,sn,o
3,2714,o,o,o,n,o
4,2715,o,o,o,sn,o


In [36]:
predict_path = "/home/vilota/mingjie/dinov3/scripts/v5/621D result/consolidated_batch_predictions.csv"
df2 = pd.read_csv(predict_path)
df2['SN'] = df2['SN'].astype(str).str.split('/').str[0].str.strip().str.upper().str.replace(r'\.0$', '', regex=True)

merged_df = pd.merge(df, df2, on='SN', how='inner')

# Compare result

In [37]:
# 1. Initialize counters for each row at 0
merged_df['number of correct "o"'] = 0
merged_df['number of incorrect "o"'] = 0

# 2. Check each position using clean vectorized comparisons
for pos in [1, 3, 5, 7, 9]:
    human_col = f'pos {pos}'          # Matches 'pos 1' from your human sheet
    pred_col = f'pos {pos} predict'   # Matches 'pos 1 predict' from the model sheet
    
    # Ensure values are lowercase and stripped for clean matching
    h_series = merged_df[human_col].astype(str).str.strip().str.lower()
    p_series = merged_df[pred_col].astype(str).str.strip().str.lower()
    
    # Model said 'o' and Human said 'o'
    merged_df['number of correct "o"'] += (p_series == 'o') & (h_series == 'o')
    
    # Model said 'o' but Human said something else (Defect)
    merged_df['number of incorrect "o"'] += (p_series == 'o') & (h_series != 'o')



In [38]:
total_correct_o = merged_df['number of correct "o"'].sum()
total_incorrect_o = merged_df['number of incorrect "o"'].sum()

# Define the exact clean 13579 column structure
rearranged_columns = [
    'SN',
    # 1. Human annotations grouped (13579)
    'pos 1', 'pos 3', 'pos 5', 'pos 7', 'pos 9',
    # 2. Model predictions and confidence rates grouped (13579)
    'pos 1 predict', 'pos 1 confidence',
    'pos 3 predict', 'pos 3 confidence',
    'pos 5 predict', 'pos 5 confidence',
    'pos 7 predict', 'pos 7 confidence',
    'pos 9 predict', 'pos 9 confidence',
    # 3. Decision metrics and counters
    'Action Required',
    'number of correct "o"',
    'number of incorrect "o"'
]

# Apply the new column sorting order to your merged DataFrame
merged_df = merged_df[rearranged_columns]

# Save the final file
merged_df.to_csv("final_evaluation_results.csv", index=False)

print(f"total number of correct 'o' predictions: {total_correct_o}")
print(f"total number of incorrect 'o' predictions: {total_incorrect_o}")

total number of correct 'o' predictions: 509
total number of incorrect 'o' predictions: 2


# Check Performance for certain baseline (all)

In [39]:
# 1. Specify your baseline threshold here (as a percentage number)
BASELINE_THRESHOLD = 75  # Change this to whatever baseline you want to test

# Ensure we don't include the TOTAL SUM row if it's already there
df_filtered = merged_df[merged_df['SN'] != 'TOTAL SUM'].copy()

total_correct_above = 0
total_incorrect_above = 0
total_dataset_predictions = 0

# 2. Loop through all positions to check predictions against the baseline
for pos in [1, 3, 5, 7, 9]:
    human_col = f'pos {pos}'
    pred_col = f'pos {pos} predict'
    conf_col = f'pos {pos} confidence'
    
    # Clean up columns (handle strings, lowercase, and convert confidence to float)
    h_series = df_filtered[human_col].astype(str).str.strip().str.lower()
    p_series = df_filtered[pred_col].astype(str).str.strip().str.lower()
    conf_series = df_filtered[conf_col].astype(str).str.replace('%', '').astype(float)
    
    # Track the grand total of ALL evaluations across the whole dataset
    total_dataset_predictions += len(df_filtered)
    
    # Create a mask for predictions that are AT or ABOVE your baseline
    above_baseline_mask = conf_series >= BASELINE_THRESHOLD
    
    # Count how many are correct vs incorrect inside this baseline group
    correct_mask = (p_series == h_series) & above_baseline_mask
    incorrect_mask = (p_series != h_series) & above_baseline_mask
    
    total_correct_above += correct_mask.sum()
    total_incorrect_above += incorrect_mask.sum()

total_above = total_correct_above + total_incorrect_above
accuracy_above_baseline = (total_correct_above / total_above * 100) if total_above > 0 else 0

# -----------------------------------------------------------
# 📊 CALCULATE AUTOMATION COVERAGE RATIO
# -----------------------------------------------------------
# Calculate what percentage of all predictions are at or above the baseline
percentage_above_baseline = (total_above / total_dataset_predictions * 100) if total_dataset_predictions > 0 else 0

# 3. Print the results to your screen
print(f"--- 📊 General Prediction Analysis for Baseline >= {BASELINE_THRESHOLD}% ---")
print(f"Total predictions above baseline:      {total_above} out of {total_dataset_predictions} total dataset samples")
print(f"Percentage of predictions above bsln:  {percentage_above_baseline:.2f}% (Workflow coverage)")
print(f"----------------------------------------------------------------------")
print(f"Number of CORRECT predictions:         {total_correct_above}")
print(f"Number of INCORRECT predictions:       {total_incorrect_above}")
print(f"Accuracy above baseline:               {accuracy_above_baseline:.2f}%")

if total_incorrect_above == 0 and (total_correct_above > 0):
    print(f"\n🎉 Success! {BASELINE_THRESHOLD}% is a safe baseline. Content above this is 100% correct.")
else:
    print(f"\n⚠️ Warning: Still found {total_incorrect_above} mistake(s) above {BASELINE_THRESHOLD}%. Raise the threshold.")

--- 📊 General Prediction Analysis for Baseline >= 75% ---
Total predictions above baseline:      511 out of 2015 total dataset samples
Percentage of predictions above bsln:  25.36% (Workflow coverage)
----------------------------------------------------------------------
Number of CORRECT predictions:         509
Number of INCORRECT predictions:       2
Accuracy above baseline:               99.61%

⚠️ Warning: Still found 2 mistake(s) above 75%. Raise the threshold.


# Check Performance for certain baseline ("o")

In [40]:
# 1. Specify your baseline threshold here (as a percentage number)
BASELINE_THRESHOLD = 90  # Change this to whatever baseline you want to test

# Ensure we don't include the TOTAL SUM row if it's already there
df_filtered = merged_df[merged_df['SN'] != 'TOTAL SUM'].copy()

total_correct_o_above = 0
total_incorrect_o_above = 0
total_dataset_predictions = 0

# 2. Loop through all positions to check predictions against the baseline
for pos in [1, 3, 5, 7, 9]:
    human_col = f'pos {pos}'
    pred_col = f'pos {pos} predict'
    conf_col = f'pos {pos} confidence'
    
    # Clean up columns (handle strings, lowercase, and convert confidence to float)
    h_series = df_filtered[human_col].astype(str).str.strip().str.lower()
    p_series = df_filtered[pred_col].astype(str).str.strip().str.lower()
    conf_series = df_filtered[conf_col].astype(str).str.replace('%', '').astype(float)
    
    # Keep track of the grand total of ALL evaluations across the whole dataset
    total_dataset_predictions += len(df_filtered)
    
    # Create masks:
    # - Prediction must be AT or ABOVE your baseline
    # - Prediction MUST specifically be 'o'
    above_baseline_mask = conf_series >= BASELINE_THRESHOLD
    predict_is_o_mask = p_series == 'o'
    
    # Combine conditions: looking strictly for high-confidence 'o' predictions
    target_mask = predict_is_o_mask & above_baseline_mask
    
    # Correct 'o': Model predicted 'o' and Human agreed it was 'o'
    correct_o_mask = (h_series == 'o') & target_mask
    
    # Incorrect 'o' (False OK / Missed Defect): Model predicted 'o' but Human flagged a defect
    incorrect_o_mask = (h_series != 'o') & target_mask
    
    total_correct_o_above += correct_o_mask.sum()
    total_incorrect_o_above += incorrect_o_mask.sum()

total_o_above = total_correct_o_above + total_incorrect_o_above
accuracy_above_baseline = (total_correct_o_above / total_o_above * 100) if total_o_above > 0 else 0

# -----------------------------------------------------------
# 📊 CALCULATE AUTOMATION COVERAGE RATIO
# -----------------------------------------------------------
# Calculate what percentage of the absolute entire dataset is automated by this threshold
percentage_automated = (total_o_above / total_dataset_predictions * 100) if total_dataset_predictions > 0 else 0

# 3. Print the results to your screen
print(f"--- 📊 'o' (OK) Prediction Analysis for Baseline >= {BASELINE_THRESHOLD}% ---")
print(f"Total 'o' predictions above baseline:  {total_o_above} out of {total_dataset_predictions} total dataset samples")
print(f"Automation Coverage Rate:              {percentage_automated:.2f}% (Percentage of total workflow cleared)")
print(f"----------------------------------------------------------------------")
print(f"Number of CORRECT 'o' predictions:     {total_correct_o_above}")
print(f"Number of INCORRECT 'o' (False OKs):   {total_incorrect_o_above}")
print(f"Precision Score above baseline:        {accuracy_above_baseline:.2f}%")

if total_incorrect_o_above == 0 and (total_correct_o_above > 0):
    print(f"\n🎉 Success! At >= {BASELINE_THRESHOLD}%, every single automated 'o' pass is 100% correct.")
else:
    print(f"\n⚠️ Warning: Found {total_incorrect_o_above} missed defect(s) allowed past your baseline! Raise the threshold.")

--- 📊 'o' (OK) Prediction Analysis for Baseline >= 90% ---
Total 'o' predictions above baseline:  511 out of 2015 total dataset samples
Automation Coverage Rate:              25.36% (Percentage of total workflow cleared)
----------------------------------------------------------------------
Number of CORRECT 'o' predictions:     509
Number of INCORRECT 'o' (False OKs):   2
Precision Score above baseline:        99.61%

⚠️ Warning: Found 2 missed defect(s) allowed past your baseline! Raise the threshold.


In [41]:
'''
# 1) Rows where the model predicted 'o' but the human label is not 'o' (incorrect 'o')
pos_list = [1, 3, 5, 7, 9]
masks = []
for pos in pos_list:
    h = merged_df[f'pos {pos}'].astype(str).str.strip().str.lower()
    p = merged_df[f'pos {pos} predict'].astype(str).str.strip().str.lower()
    masks.append((p == 'o') & (h != 'o'))

incorrect_o_mask = masks[0].copy() if masks else merged_df.index.to_series().apply(lambda _: False)
for m in masks[1:]:
    incorrect_o_mask |= m
df_incorrect_o = merged_df[incorrect_o_mask].copy()

print(f"Incorrect 'o' predictions (model=='o' but human!= 'o'): {len(df_incorrect_o)}")
if not df_incorrect_o.empty:
    display(df_incorrect_o)

# 2) Rows where the model predicted 'n' OR the human ground truth contains 'mn' or 'xn'
pred_n_masks = []
h_mn_xn_masks = []
for pos in pos_list:
    h = merged_df[f'pos {pos}'].astype(str).str.strip().str.lower()
    p = merged_df[f'pos {pos} predict'].astype(str).str.strip().str.lower()
    pred_n_masks.append(p == 'n')
    h_mn_xn_masks.append(h.str.contains('mn|xn', regex=True))

pred_n_mask = pred_n_masks[0].copy() if pred_n_masks else merged_df.index.to_series().apply(lambda _: False)
for m in pred_n_masks[1:]:
    pred_n_mask |= m

h_mn_xn_mask = h_mn_xn_masks[0].copy() if h_mn_xn_masks else merged_df.index.to_series().apply(lambda _: False)
for m in h_mn_xn_masks[1:]:
    h_mn_xn_mask |= m

combined_mask = pred_n_mask | h_mn_xn_mask
df_pred_n_or_mn_xn = merged_df[combined_mask].copy()

print(f"Rows where prediction=='n' OR human contains 'mn'/'xn': {len(df_pred_n_or_mn_xn)}")
if not df_pred_n_or_mn_xn.empty:
    display(df_pred_n_or_mn_xn)

# Optional: save these subsets for inspection
df_incorrect_o.to_csv('incorrect_o_rows.csv', index=False)
df_pred_n_or_mn_xn.to_csv('pred_n_or_mn_xn_rows.csv', index=False)
print("Saved CSV: incorrect_o_rows.csv, pred_n_or_mn_xn_rows.csv")
'''

'\n# 1) Rows where the model predicted \'o\' but the human label is not \'o\' (incorrect \'o\')\npos_list = [1, 3, 5, 7, 9]\nmasks = []\nfor pos in pos_list:\n    h = merged_df[f\'pos {pos}\'].astype(str).str.strip().str.lower()\n    p = merged_df[f\'pos {pos} predict\'].astype(str).str.strip().str.lower()\n    masks.append((p == \'o\') & (h != \'o\'))\n\nincorrect_o_mask = masks[0].copy() if masks else merged_df.index.to_series().apply(lambda _: False)\nfor m in masks[1:]:\n    incorrect_o_mask |= m\ndf_incorrect_o = merged_df[incorrect_o_mask].copy()\n\nprint(f"Incorrect \'o\' predictions (model==\'o\' but human!= \'o\'): {len(df_incorrect_o)}")\nif not df_incorrect_o.empty:\n    display(df_incorrect_o)\n\n# 2) Rows where the model predicted \'n\' OR the human ground truth contains \'mn\' or \'xn\'\npred_n_masks = []\nh_mn_xn_masks = []\nfor pos in pos_list:\n    h = merged_df[f\'pos {pos}\'].astype(str).str.strip().str.lower()\n    p = merged_df[f\'pos {pos} predict\'].astype(str).s

In [42]:
# Save rows where the predictions contain at least one 'n'
# or at least one 'sn' prediction with confidence >= 45%
pos_list = [1, 3, 5, 7, 9]

paired_cols = [
    (f'pos {pos} predict', f'pos {pos} confidence')
    for pos in pos_list
    if f'pos {pos} predict' in merged_df.columns and f'pos {pos} confidence' in merged_df.columns
]
if not paired_cols:
    raise ValueError("No matching prediction/confidence column pairs found in merged_df.")

pred_frame = pd.DataFrame({pred_col: merged_df[pred_col].astype(str).str.strip().str.lower() for pred_col, _ in paired_cols})
conf_frame = pd.DataFrame({conf_col: pd.to_numeric(merged_df[conf_col].astype(str).str.replace('%', '', regex=False).str.strip(), errors='coerce') for _, conf_col in paired_cols})

has_n_mask = pred_frame.eq('n').any(axis=1)

# any single 'sn' with confidence >= 45%
any_sn_45_mask = pd.DataFrame({
    pred_col: (pred_frame[pred_col] == 'sn') & (conf_frame[conf_col] >= 45)
    for pred_col, conf_col in paired_cols
}).any(axis=1)

selected_mask = has_n_mask | any_sn_45_mask
saved_df = merged_df[selected_mask].copy()

output_path = 'predictions_with_n_or_four_high_conf_sn.csv'
saved_df.to_csv(output_path, index=False)

print(f"Saved {len(saved_df)} rows to {output_path}")
print(f"Rows with at least one 'n': {has_n_mask.sum()}")
print(f"Rows with any 'sn' at confidence >= 45%: {any_sn_45_mask.sum()}")

Saved 129 rows to predictions_with_n_or_four_high_conf_sn.csv
Rows with at least one 'n': 15
Rows with any 'sn' at confidence >= 45%: 127
